# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umerkang66/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook establishes the formal **Data Contract** for **Lane 2: Refresh / Content Opportunity Scoring**. It defines the precise grain, tables, time boundaries, feature classifications, and target proxy for the warehouse release (~79M rows hosted on Hugging Face). Every contractual claim is empirically verified with DuckDB SQL queries over mid-panel month partition `month=2026-03`, followed by our first 5 honest features and an explicit demonstration and elimination of the target leakage trap.

> **Skills loaded:** `writing-data-contracts` + `flyrank/flyrank-data`


In [1]:
# Setup: Environment, Authentication & DuckDB Connection
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# Load HF token securely (env var -> Colab secret -> local .env; never print secrets)
try:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path=r'../../.env')
    load_dotenv(dotenv_path=r'../.env')
    load_dotenv(dotenv_path=r'.env')
except Exception:
    pass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Resolve dataset locations (use local Hugging Face cache if present, fallback to remote hf://)
REL = 'hf://datasets/FlyRank/internship-warehouse'

def resolve_table_path(subpath, filename):
    try:
        from huggingface_hub import hf_hub_download
        cached = hf_hub_download(
            repo_id='FlyRank/internship-warehouse',
            filename=filename,
            repo_type='dataset',
            token=HF_TOKEN
        ).replace('\\', '/')
        return f"read_parquet('{cached}')"
    except Exception:
        return f"read_parquet('{REL}/{subpath}')"

SRC_FACT_MARCH = resolve_table_path(
    'fact_content_daily_performance/month=2026-03/*.parquet',
    'fact_content_daily_performance/month=2026-03/data_0.parquet'
)
SRC_DIM_CLIENTS = resolve_table_path('dim_clients.parquet', 'dim_clients.parquet')
SRC_DIM_CONTENT = resolve_table_path('dim_content.parquet', 'dim_content.parquet')

print("DuckDB connected. Warehouse tables mapped successfully.")

DuckDB connected. Warehouse tables mapped successfully.


## 1. Unit of analysis + time window

### Plain-Words Contract Answers (1–3)

1. **What one row means for your lane (Unit of Analysis / Grain):**
   - **In the Warehouse Daily Fact Table:** One row represents **one daily performance record for a single pseudonymized content item on a specific client domain: `report_date × client_hash_id × content_hash_id`**.
   - **In the Decision / Modeling Feature Frame:** One row represents **one unique content asset (`client_hash_id × content_hash_id`)** evaluated at a discrete monthly editorial decision moment.

2. **Which table(s) you'll use:**
   - `fact_content_daily_performance` (partitioned by month; strictly developing on mid-panel month partition `month=2026-03` to keep the final month June 2026 sealed as a pristine holdout test window).
   - `dim_clients` (for access profiles, client history start dates, and client-level holdout splitting).
   - `dim_content` (for article properties including `content_type`, `search_volume`, `competition`, and creation dates).

3. **Which time window:**
   - For our mid-panel contract verification on `month=2026-03` (2026-03-01 through 2026-03-31):
     - **Observation / Feature Window (Days 1–20):** `2026-03-01` to `2026-03-20` (20 days). All model input features must be aggregated strictly within this window.
     - **Decision Moment:** End of day `2026-03-20` (the exact point in time when an SEO manager or editor decides which content items to queue for refresh).
     - **Outcome / Label Window (Days 21–31):** `2026-03-21` to `2026-03-31` (11 days). Future traffic velocity and position changes are observed here to establish ground truth.


In [2]:
# Verification of Table Metadata & Client Overview
client_summary = con.sql(f"""
    SELECT 
        COUNT(*) AS total_clients,
        COUNT(*) FILTER (WHERE has_gsc_access IS TRUE) AS clients_with_gsc,
        COUNT(*) FILTER (WHERE has_ga4_access IS TRUE) AS clients_with_ga4,
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start
    FROM {SRC_DIM_CLIENTS}
""").df()

print("dim_clients Table Overview:")
display(client_summary)

dim_clients Table Overview:


,total_clients,clients_with_gsc,clients_with_ga4,earliest_gsc_start,latest_gsc_start
0,104,67,54,2025-01-27,2026-06-02


## 2. Fields: feature / label / context / excluded

### Plain-Words Contract Answers (4–5)

4. **What you'd predict or rank (label or proxy):**
   - **Target Proxy (`is_declining`):** Whether a content item with actionable baseline search demand (`obs_impressions >= 50` during the 20-day observation window) experiences an organic traffic drop of **> 20% in average daily search impressions** during the forward 11-day outcome window compared to its observation baseline. This continuous opportunity score / binary proxy identifies high-demand pages losing search traction that warrant an editorial refresh.

5. **One thing you deliberately exclude (and why):**
   - **Deliberately Excluded:** `outcome_impressions` (and any metric calculated from or within the outcome window, including `trend_direction` and `trend_pct`).
   - **Why:** These metrics occur _after_ the decision cutoff date (2026-03-20). Including post-cutoff metrics in the feature vector is the classic data leakage trap demonstrated in Notebook 02: the model achieves a trivial ~0.99 ROC-AUC score during training but fails entirely when deployed to real future data. We also exclude raw pseudonym IDs (`client_hash_id`, `content_hash_id`) from feature vectors (reserving them strictly for grouping and splits) because learning hash patterns produces memorization rather than generalizable signals.

### Formal Field Classification Matrix

| Field Name                      | Category          | Knowable Moment          | Rationale & Definition                                                                        |
| ------------------------------- | ----------------- | ------------------------ | --------------------------------------------------------------------------------------------- |
| `feat_obs_impressions`          | **Feature**       | Decision Moment (Day 20) | Cumulative search impressions logged in GSC across days 1–20.                                 |
| `feat_obs_clicks`               | **Feature**       | Decision Moment (Day 20) | Cumulative organic search clicks logged across days 1–20.                                     |
| `feat_obs_ctr`                  | **Feature**       | Decision Moment (Day 20) | Observation CTR (`clicks / impressions * 100`). Measures user click propensity.               |
| `feat_obs_avg_position`         | **Feature**       | Decision Moment (Day 20) | Mean GSC rank across days 1–20 (lower number = closer to page 1).                             |
| `feat_obs_active_days`          | **Feature**       | Decision Moment (Day 20) | Distinct days with >= 1 impression during days 1–20. Measures demand consistency.             |
| `is_declining`                  | **Label / Proxy** | Post-Outcome (Day 31)    | Binary ground-truth: 1 if outcome daily rate < 0.80 \* observation daily rate.                |
| `client_hash_id`                | **Context**       | Pre-Analysis             | Pseudonym client identifier; strictly for client-holdout splits (`GroupShuffleSplit`).        |
| `content_hash_id`               | **Context**       | Pre-Analysis             | Unique content item identifier; strictly for grouping, joining, and audit logs.               |
| `report_date`                   | **Context**       | Ingestion                | Event timestamp defining observation vs. outcome temporal partitions.                         |
| `outcome_impressions`           | **Excluded**      | Future (Days 21–31)      | Post-decision data. Strictly excluded from features to prevent leakage trap.                  |
| `trend_direction` / `trend_pct` | **Excluded**      | Future Outcome Window    | Derived from comparison windows overlapping the target; label source only.                    |
| `ga4_*` (unflagged)             | **Excluded**      | Zero-Filled / Gaps       | GA4 columns are zero-filled before client start dates; requires `ga4_data_available IS TRUE`. |


In [3]:
# Inspect Content Metadata Context & Missingness Structure
content_summary = con.sql(f"""
    SELECT 
        COUNT(*) AS total_content_items,
        COUNT(DISTINCT client_hash_id) AS distinct_clients,
        COUNT(DISTINCT content_type) AS distinct_content_types,
        ROUND(AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0.0 END) * 100.0, 1) AS pct_missing_word_count,
        ROUND(AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0.0 END) * 100.0, 1) AS pct_missing_search_volume
    FROM {SRC_DIM_CONTENT}
""").df()

print("dim_content Context & Missingness Overview:")
display(content_summary)

dim_content Context & Missingness Overview:


,total_content_items,distinct_clients,distinct_content_types,pct_missing_word_count,pct_missing_search_volume
0,519606,84,3,34.2,27.4


## 3. Verify it with queries (grain, counts, missing values, windows)

### Part Two: Prove Three Facts with Three Queries on Mid-Panel Month (`month=2026-03`)

1. **Query 1 — The Grain:** Prove that one row in the daily performance fact table really is uniquely identified by `(report_date, client_hash_id, content_hash_id)` (`HAVING COUNT(*) > 1` returns 0 rows).
2. **Query 2 — Row Count & Date Span:** Measure the exact row count, unique entity counts, and `MIN`/`MAX` report dates for `month=2026-03`.
3. **Query 3 — Data Availability:** Filter with `IS TRUE` on `gsc_data_available` and `ga4_data_available` to reveal exact survival rates across the monthly panel.


In [4]:
# ==============================================================================
# QUERY 1: THE GRAIN VERIFICATION
# ==============================================================================
q1_grain = con.sql(f"""
    SELECT 
        report_date, 
        client_hash_id, 
        content_hash_id, 
        COUNT(*) AS duplicate_row_count
    FROM {SRC_FACT_MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING duplicate_row_count > 1
    LIMIT 5
""").df()

print("=== QUERY 1: Grain Verification (Duplicate Violations) ===")
print(f"Duplicate rows found: {len(q1_grain)}")
if len(q1_grain) == 0:
    print("VERDICT: Grain strictly confirmed — composite key (report_date x client x content) has ZERO duplicates.\n")
else:
    display(q1_grain)

# ==============================================================================
# QUERY 2: ROW COUNT AND DATE SPAN VERIFICATION
# ==============================================================================
q2_span = con.sql(f"""
    SELECT 
        COUNT(*) AS total_daily_rows,
        COUNT(DISTINCT client_hash_id) AS active_clients,
        COUNT(DISTINCT content_hash_id) AS active_content_items,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM {SRC_FACT_MARCH}
""").df()

print("=== QUERY 2: Slice Row Count and Date Span (month=2026-03) ===")
display(q2_span)
print(f"VERDICT: Exactly {q2_span['total_daily_rows'][0]:,} rows spanning from "
      f"{q2_span['min_report_date'][0]} to {q2_span['max_report_date'][0]} across {q2_span['active_clients'][0]} clients.\n")

# ==============================================================================
# QUERY 3: DATA AVAILABILITY AUDIT (FILTERED WITH IS TRUE)
# ==============================================================================
q3_availability = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS full_stack_available_rows,
        ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS pct_gsc_surviving,
        ROUND(COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS pct_ga4_surviving,
        ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS pct_full_stack_surviving
    FROM {SRC_FACT_MARCH}
""").df()
8
print("=== QUERY 3: Data Availability Audit (Filtered with IS TRUE) ===")
display(q3_availability)
print(f"VERDICT: {q3_availability['gsc_available_rows'][0]:,} rows ({q3_availability['pct_gsc_surviving'][0]}%) survive GSC filtering. "
      f"Only {q3_availability['ga4_available_rows'][0]:,} rows ({q3_availability['pct_ga4_surviving'][0]}%) have GA4 tracking enabled. "
      f"Exactly {q3_availability['full_stack_available_rows'][0]:,} rows ({q3_availability['pct_full_stack_surviving'][0]}%) survive dual tracking.")

=== QUERY 1: Grain Verification (Duplicate Violations) ===
Duplicate rows found: 0
VERDICT: Grain strictly confirmed — composite key (report_date x client x content) has ZERO duplicates.

=== QUERY 2: Slice Row Count and Date Span (month=2026-03) ===


,total_daily_rows,active_clients,active_content_items,min_report_date,max_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


VERDICT: Exactly 9,841,378 rows spanning from 2026-03-01 00:00:00 to 2026-03-31 00:00:00 across 55 clients.

=== QUERY 3: Data Availability Audit (Filtered with IS TRUE) ===


,total_rows,gsc_available_rows,ga4_available_rows,full_stack_available_rows,pct_gsc_surviving,pct_ga4_surviving,pct_full_stack_surviving
0,9841378,3611061,413966,364347,36.69,4.21,3.7


VERDICT: 3,611,061 rows (36.69%) survive GSC filtering. Only 413,966 rows (4.21%) have GA4 tracking enabled. Exactly 364,347 rows (3.7%) survive dual tracking.


### Part Three: Five Features, Max (Mid-Panel Month `month=2026-03`)

We extract a concise, high-signal feature vector where **every single feature has an unambiguous "knowable at the decision moment because..." justification**:

1. `feat_obs_impressions` — **Knowable at the decision moment because** total organic search impressions recorded in GSC logs during days 1–20 are historical query events logged before the review cutoff date.
2. `feat_obs_clicks` — **Knowable at the decision moment because** search clicks accrued during the first 20 days occurred and were captured in Google Search Console prior to the decision point.
3. `feat_obs_ctr` — **Knowable at the decision moment because** click-through rate is calculated strictly as historical clicks divided by historical impressions within the observation window.
4. `feat_obs_avg_position` — **Knowable at the decision moment because** average search ranking position is computed from GSC search result appearances that already occurred during days 1–20.
5. `feat_obs_active_days` — **Knowable at the decision moment because** the count of distinct days with active search visibility during days 1–20 is fully observable before the decision cutoff.


In [5]:
# Feature Frame Extraction Query in DuckDB
feature_extraction_sql = f"""
WITH content_windowed AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        -- Observation Window: Days 1–20 (2026-03-01 to 2026-03-20)
        SUM(CASE WHEN report_date <= DATE '2026-03-20' THEN gsc_impressions ELSE 0 END) AS obs_impressions,
        SUM(CASE WHEN report_date <= DATE '2026-03-20' THEN gsc_clicks ELSE 0 END) AS obs_clicks,
        AVG(CASE WHEN report_date <= DATE '2026-03-20' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS obs_avg_pos,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-20' AND gsc_impressions > 0 THEN report_date ELSE NULL END) AS obs_active_days,
        
        -- Outcome Window: Days 21–31 (2026-03-21 to 2026-03-31)
        SUM(CASE WHEN report_date > DATE '2026-03-20' THEN gsc_impressions ELSE 0 END) AS outcome_impressions
    FROM {SRC_FACT_MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING obs_impressions >= 50
)
SELECT 
    client_hash_id,
    content_hash_id,
    -- 5 Honest Features (strictly knowable at decision cutoff 2026-03-20)
    obs_impressions AS feat_obs_impressions,
    obs_clicks AS feat_obs_clicks,
    ROUND(obs_clicks * 100.0 / obs_impressions, 3) AS feat_obs_ctr,
    ROUND(COALESCE(obs_avg_pos, 0.0), 2) AS feat_obs_avg_position,
    obs_active_days AS feat_obs_active_days,
    
    -- Target Proxy: Drop > 20% in average daily search impressions
    -- Observation daily rate = obs_impressions / 20.0; Outcome daily rate = outcome_impressions / 11.0
    CASE 
        WHEN (outcome_impressions / 11.0) < 0.80 * (obs_impressions / 20.0) THEN 1 
        ELSE 0 
    END AS is_declining,
    
    -- Deliberate Leak Column: Future outcome impressions for Step 4 trap experiment
    outcome_impressions AS leak_outcome_impressions
FROM content_windowed
"""

print("Extracting 5 features + target proxy + deliberate leak column from DuckDB...")
df_features = con.sql(feature_extraction_sql).df()

print(f"Extracted feature frame: {df_features.shape[0]:,} content items x {df_features.shape[1]} columns.")
print("\nClass Distribution (is_declining):\n", df_features['is_declining'].value_counts(normalize=True).rename(lambda x: f"Class {x}").apply(lambda x: f"{x*100:.2f}%"))
display(df_features.head())

Extracting 5 features + target proxy + deliberate leak column from DuckDB...
Extracted feature frame: 102,537 content items x 9 columns.

Class Distribution (is_declining):
 is_declining
Class 0    67.62%
Class 1    32.38%
Name: proportion, dtype: object


,client_hash_id,content_hash_id,feat_obs_impressions,feat_obs_clicks,feat_obs_ctr,feat_obs_avg_position,feat_obs_active_days,is_declining,leak_outcome_impressions
0,client_ff644d8251367cbb,content_942d977473471b3b,213.0,0.0,0.000,7.95,20,0,150.0
1,client_ff644d8251367cbb,content_5bb4b369b65d029a,115.0,1.0,0.870,16.06,18,1,17.0
2,client_ff644d8251367cbb,content_481e74bdc286c192,484.0,0.0,0.000,30.78,20,0,255.0
3,client_ff644d8251367cbb,content_1cd21738e59ebab5,1550.0,11.0,0.710,4.38,20,1,656.0
4,client_ff644d8251367cbb,content_73153267e2b7b9bc,477.0,2.0,0.419,4.34,20,1,140.0


### Part Four: The Trap — Deliberate Leak Experiment & Honest Recovery

In Notebook 02, we saw how including label-derived or future columns creates a catastrophic illusion of predictive accuracy. Here, we perform this lesson on **real warehouse data**:

1. **Spring the Trap:** We train a Random Forest model using the 5 honest features **plus ONE future outcome column** (`leak_outcome_impressions`). We observe the quick score jump toward near perfection (~0.99 ROC-AUC).
2. **Eliminate the Trap:** We permanently delete the leaked column (`df_features.drop(columns=['leak_outcome_impressions'])`).
3. **Retain the Honest Score:** We re-train the model strictly on the 5 pre-decision features and report the honest, realistic baseline number (~0.66 ROC-AUC).


In [6]:
# The Leakage Trap Experiment: Spring the trap, observe perfection, delete it, keep honest score
honest_feature_cols = [
    'feat_obs_impressions', 
    'feat_obs_clicks', 
    'feat_obs_ctr', 
    'feat_obs_avg_position', 
    'feat_obs_active_days'
]
leaked_feature_cols = honest_feature_cols + ['leak_outcome_impressions']

y = df_features['is_declining']

# Stratified train/test split (75/25)
train_idx, test_idx = train_test_split(
    df_features.index, 
    test_size=0.25, 
    random_state=42, 
    stratify=y
)

# Step 1: Model with deliberate leakage
X_train_leak = df_features.loc[train_idx, leaked_feature_cols]
X_test_leak  = df_features.loc[test_idx, leaked_feature_cols]
y_train      = y.loc[train_idx]
y_test       = y.loc[test_idx]

rf_leaked = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_leaked.fit(X_train_leak, y_train)

probs_leaked = rf_leaked.predict_proba(X_test_leak)[:, 1]
preds_leaked = rf_leaked.predict(X_test_leak)
auc_leaked = roc_auc_score(y_test, probs_leaked)
acc_leaked = accuracy_score(y_test, preds_leaked)

print("=" * 65)
print("EXPERIMENT 1: MODEL WITH DELIBERATE LEAK (THE TRAP)")
print(f"Features: {leaked_feature_cols}")
print(f"ROC-AUC Score: {auc_leaked:.4f}  <-- Catastrophic target leakage!")
print(f"Accuracy:      {acc_leaked:.4f}")
print("=" * 65)

# Step 2: Delete the leak column permanently
print("\n--> Permanently dropping leaked column 'leak_outcome_impressions' from dataset...")
df_features = df_features.drop(columns=['leak_outcome_impressions'])

# Step 3: Train honest model on 5 valid features only
X_train_honest = df_features.loc[train_idx, honest_feature_cols]
X_test_honest  = df_features.loc[test_idx, honest_feature_cols]

rf_honest = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_honest.fit(X_train_honest, y_train)

probs_honest = rf_honest.predict_proba(X_test_honest)[:, 1]
preds_honest = rf_honest.predict(X_test_honest)
auc_honest = roc_auc_score(y_test, probs_honest)
acc_honest = accuracy_score(y_test, preds_honest)

print("=" * 65)
print("EXPERIMENT 2: HONEST MODEL (LEAK ELIMINATED)")
print(f"Features: {honest_feature_cols}")
print(f"ROC-AUC Score: {auc_honest:.4f}  <-- Honest, production-safe score")
print(f"Accuracy:      {acc_honest:.4f}")
print("=" * 65)

print("\nClassification Report (Honest Model):")
print(classification_report(y_test, preds_honest, digits=3))

trap_comparison = pd.DataFrame({
    'Experiment': ['With Deliberate Leak', 'Honest Model (Leak Dropped)'],
    'Features Count': [len(leaked_feature_cols), len(honest_feature_cols)],
    'ROC-AUC': [f"{auc_leaked:.4f}", f"{auc_honest:.4f}"],
    'Accuracy': [f"{acc_leaked:.4f}", f"{acc_honest:.4f}"],
    'Validity': ['INVALID (Future Information)', 'VALID (Production Ready)']
})
display(trap_comparison)

EXPERIMENT 1: MODEL WITH DELIBERATE LEAK (THE TRAP)
Features: ['feat_obs_impressions', 'feat_obs_clicks', 'feat_obs_ctr', 'feat_obs_avg_position', 'feat_obs_active_days', 'leak_outcome_impressions']
ROC-AUC Score: 0.9900  <-- Catastrophic target leakage!
Accuracy:      0.9311

--> Permanently dropping leaked column 'leak_outcome_impressions' from dataset...
EXPERIMENT 2: HONEST MODEL (LEAK ELIMINATED)
Features: ['feat_obs_impressions', 'feat_obs_clicks', 'feat_obs_ctr', 'feat_obs_avg_position', 'feat_obs_active_days']
ROC-AUC Score: 0.6654  <-- Honest, production-safe score
Accuracy:      0.6796

Classification Report (Honest Model):
              precision    recall  f1-score   support

           0      0.684     0.977     0.805     17335
           1      0.550     0.058     0.105      8300

    accuracy                          0.680     25635
   macro avg      0.617     0.518     0.455     25635
weighted avg      0.641     0.680     0.578     25635



,Experiment,Features Count,ROC-AUC,Accuracy,Validity
0,With Deliberate Leak,6,0.9900,0.9311,INVALID (Future Information)
1,Honest Model (Leak Dropped),5,0.6654,0.6796,VALID (Production Ready)


## 4. Data limits

### Named Limitations of this Data Slice

1. **Severe GA4 Availability Drop & Zero-Filling (The Primary Slice Limit):**
   In this mid-panel slice (`month=2026-03`), Google Search Console data is available for 36.7% of rows (`gsc_data_available IS TRUE`), but Google Analytics 4 is available for only **4.2% of rows** (`ga4_data_available IS TRUE`). Furthermore, for rows prior to a client's `ga4_data_start`, GA4 metrics (`ga4_pageviews`, `ga4_sessions`, `ga4_engaged_sessions`) are **zero-filled with `ga4_data_available = FALSE`** rather than represented as `NULL`. Any naive aggregation or model that fails to filter with `IS TRUE` will conflate uninstrumented tracking with zero user interest.

2. **Unbalanced History Across Clients (Panel Imbalance):**
   Client onboarding dates vary widely (`gsc_data_start` ranges from 2025-01-27 to recent 2026 dates). A static calendar window inevitably penalizes recently joined clients with truncated history.

3. **Truncated Outcome Window in Single-Month Slice:**
   Because this mid-panel slice is isolated to March 2026, the forward outcome window is truncated to 11 days (March 21–31) to avoid leaking April data. At full warehouse production scale, a 60- or 90-day observation window predicting a full 30-day forward outcome is preferred.


In [7]:
# Empirical Demonstration of the Primary Data Limitation: GA4 Zero-Fill vs Availability Flag
limit_audit = con.sql(f"""
    SELECT 
        ga4_data_available,
        COUNT(*) AS row_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_rows,
        COUNT(*) FILTER (WHERE ga4_pageviews = 0) AS zero_pageviews_count,
        ROUND(COUNT(*) FILTER (WHERE ga4_pageviews = 0) * 100.0 / COUNT(*), 2) AS pct_zero_pageviews
    FROM {SRC_FACT_MARCH}
    GROUP BY ga4_data_available
""").df()

print("=== Demonstration of Slice Limitation: GA4 False Zeros ===")
display(limit_audit)
print("TAKEAWAY: Notice that for ga4_data_available = FALSE, 100% of rows have ga4_pageviews = 0. "
      "These are synthetic zero-fills, NOT zero engagement. Always check IS TRUE before using GA4.")

=== Demonstration of Slice Limitation: GA4 False Zeros ===


,ga4_data_available,row_count,pct_of_rows,zero_pageviews_count,pct_zero_pageviews
0,False,6408671,65.12,6408671,100.00
1,<NA>,3018741,30.67,0,0.00
2,True,413966,4.21,649,0.16


TAKEAWAY: Notice that for ga4_data_available = FALSE, 100% of rows have ga4_pageviews = 0. These are synthetic zero-fills, NOT zero engagement. Always check IS TRUE before using GA4.


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
